# One Request, Two Machines: Simulating Prefill/Decode Disaggregation

Companion notebook for the Oct 19, 2026 post. A tiny discrete-time queueing sim (seed 42) compares **colocated** serving — one GPU does prefill and decode — against **disaggregated** serving — a prefill pool and a decode pool joined by a KV-cache transfer.

Hardware model is illustrative (1B-class dense model, A100-class FLOPs and HBM bandwidth); the workload is agentic (16K–64K-token prompts, short answers). This is a mechanism demo, not a serving benchmark.

In [ ]:
"""Toy discrete-time simulation: colocated vs disaggregated prefill/decode serving.

Illustrative only — a 1B-class model on A100-class hardware, clearly-labeled toy
constants. Purpose: demonstrate the MECHANISM (prefill stalls decode on a shared
GPU; separation removes the stalls at the cost of one KV-cache transfer).
Not a serving benchmark.
"""
import numpy as np

SEED = 42
DT = 1.0            # tick, ms
FLOPS = 312e12      # FP16 dense, A100-class
WEIGHT_GB = 2.0     # 1B model, FP16
BW_GBS = 2000.0     # HBM bandwidth GB/s
T_KV = 10.0         # KV-cache transfer prefill->decode, ms (illustrative)

def prefill_ms(prompt_len):
    return 2 * 1e9 * prompt_len / FLOPS * 1000.0   # compute-bound

def run(lam, t_arr=60_000.0, seed=SEED):
    rng = np.random.default_rng(seed)
    n = int(lam * t_arr / 1000 * 1.6) + 8
    ia = rng.exponential(1000.0 / lam, size=n)
    arrivals = np.cumsum(ia)
    arrivals = arrivals[arrivals < t_arr]
    prompts = rng.integers(16000, 64001, size=len(arrivals))  # agentic: long prompts
    outs = rng.integers(20, 61, size=len(arrivals))

    out = {}
    for cfg in ("coloc", "disagg"):
        # prefill state
        pq = []                      # (arrive_t, Lp, Lo)
        pre = None                   # [remaining_ms, arrive_t, Lp, Lo]
        # decode state
        dq = []                      # waiting for decode slot
        active = []                  # [remain_tokens, last_emit, progress, arrive_t, ttft, itls]
        itls = []
        ttfts = []
        transfer_frac = []
        t = 0.0
        ai = 0
        done = 0
        total = len(arrivals)
        t_end = t_arr + 120_000.0
        while done < total and t < t_end:
            while ai < total and arrivals[ai] <= t:
                pq.append((arrivals[ai], prompts[ai], outs[ai]))
                ai += 1
            # ---- prefill progress ----
            if pre is None and pq:
                a, lp, lo = pq.pop(0)
                pre = [prefill_ms(lp), a, lp, lo]
            if pre is not None:
                # colocated: prefill owns the GPU while it runs (prefill-priority)
                pre[0] -= DT
                if pre[0] <= 0:
                    _, a, lp, lo = pre
                    pre = None
                    if cfg == "disagg":
                        dq.append((a, lo, t + T_KV))   # KV transfer delay
                    else:
                        active.append([lo, t, 0.0, a, None, []])
            # disaggregated: prefill GPU works independently of decode GPU
            if cfg == "disagg":
                # (prefill handled above; decode runs every tick below)
                pass
            # ---- decode progress ----
            if cfg == "coloc" and pre is not None:
                pass  # GPU busy with prefill: decode stalls (the interference)
            else:
                while dq and dq[0][2] <= t:
                    a, lo, _ = dq.pop(0)
                    active.append([lo, t, 0.0, a, None, []])
                b = len(active)
                if b:
                    # aggregate decode is bandwidth-bound: reloading WEIGHT_GB per
                    # step shared across b streams -> per-stream token time = b*W/BW
                    tok_ms = b * WEIGHT_GB / BW_GBS * 1000.0
                    for s in active:
                        s[2] += DT / tok_ms
                    keep = []
                    for s in active:
                        if s[2] >= 1.0:
                            s[2] -= 1.0
                            s[0] -= 1
                            if s[4] is None:
                                s[4] = t - s[3]          # TTFT
                                if cfg == "disagg":
                                    transfer_frac.append(T_KV / max(s[4], 1e-6))
                            else:
                                itl = t - s[1]
                                s[5].append(itl)
                                itls.append(itl)
                            s[1] = t
                        if s[0] > 0:
                            keep.append(s)
                        else:
                            done += 1
                            ttfts.append(s[4])
                    active = keep
            t += DT
        itls = np.array(itls)
        ttfts = np.array(ttfts)
        out[cfg] = {
            "p50_itl": float(np.percentile(itls, 50)) if len(itls) else float("nan"),
            "p90_itl": float(np.percentile(itls, 90)) if len(itls) else float("nan"),
            "p99_itl": float(np.percentile(itls, 99)) if len(itls) else float("nan"),
            "max_itl": float(itls.max()) if len(itls) else float("nan"),
            "tok_slo": float(np.mean(itls <= 20.0)) if len(itls) else float("nan"),
            "p90_ttft": float(np.percentile(ttfts, 90)) if len(ttfts) else float("nan"),
            "transfer_frac": float(np.mean(transfer_frac)) if transfer_frac else 0.0,
            "n_tok": len(itls),
        }
    return out


## The interference, measured

Sweep the arrival rate and compare tail streaming latency (p99 inter-token latency), worst-case stalls (max ITL), and time-to-first-token (p90 TTFT).

In [ ]:
for lam in (0.5, 1.0, 1.5, 2.0):
    r = run(lam); c, d = r["coloc"], r["disagg"]
    print(f"λ={lam}/s  coloc: p99 ITL {c['p99_itl']:7.1f}ms  max ITL {c['max_itl']:7.0f}ms  p90 TTFT {c['p90_ttft']:5.0f}ms")
    print(f"          disagg: p99 ITL {d['p99_itl']:7.1f}ms  max ITL {d['max_itl']:7.0f}ms  p90 TTFT {d['p90_ttft']:5.0f}ms  "
          f"KV-xfer/TTFT {d['transfer_frac']:.3f}")


## The assertions: what separation buys

In [ ]:
r = run(1.0); c, d = r["coloc"], r["disagg"]
assert 3.0 * d["p99_itl"] < c["p99_itl"], "disagg must cut p99 ITL 3x+"
assert 10.0 * d["max_itl"] < c["max_itl"], "coloc must show stall spikes in max ITL"
assert d["p90_ttft"] < c["p90_ttft"], "disagg must cut p90 TTFT"
assert d["transfer_frac"] < 0.20, "KV transfer must stay a small TTFT fraction"
print("ALL ASSERTIONS PASS")
print(f"headline (λ=1.0/s): p99 ITL {c['p99_itl']:.1f}ms → {d['p99_itl']:.1f}ms "
      f"({c['p99_itl']/d['p99_itl']:.0f}x); p90 TTFT {c['p90_ttft']:.0f}ms → {d['p90_ttft']:.0f}ms; "
      f"max ITL {c['max_itl']:.0f}ms → {d['max_itl']:.0f}ms")


## The chart

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

lams = [0.5, 1.0, 1.5, 2.0]
coloc_p99, disagg_p99, coloc_ttft, disagg_ttft = [], [], [], []
for lam in lams:
    r = run(lam)
    coloc_p99.append(r["coloc"]["p99_itl"]); disagg_p99.append(r["disagg"]["p99_itl"])
    coloc_ttft.append(r["coloc"]["p90_ttft"]); disagg_ttft.append(r["disagg"]["p90_ttft"])

TEAL, ORANGE = "#0f766e", "#c2703d"
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
ax[0].plot(lams, coloc_p99, "o-", color=ORANGE, lw=2.5, ms=8, label="colocated (shared GPU)")
ax[0].plot(lams, disagg_p99, "s-", color=TEAL, lw=2.5, ms=8, label="disaggregated (P/D pools)")
ax[0].set_yscale("log"); ax[0].set_xlabel("arrival rate (req/s)"); ax[0].set_ylabel("p99 inter-token latency (ms, log)")
ax[0].set_title("Tail streaming latency: stalls vanish when phases split"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(lams, coloc_ttft, "o-", color=ORANGE, lw=2.5, ms=8, label="colocated (shared GPU)")
ax[1].plot(lams, disagg_ttft, "s-", color=TEAL, lw=2.5, ms=8, label="disaggregated (P/D pools)")
ax[1].set_xlabel("arrival rate (req/s)"); ax[1].set_ylabel("p90 time-to-first-token (ms)")
ax[1].set_title("First-token latency under load"); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.suptitle("Toy queueing sim: 1B-class model, 16K-64K prompts, seed 42 (illustrative, not a benchmark)",
             fontsize=10, color="#5e6a75")
fig.tight_layout()
fig.savefig("publish-kit-2026-10-19-diagrams/sim_tail_latency.png", dpi=150)
print("saved publish-kit-2026-10-19-diagrams/sim_tail_latency.png")
print(f"at 2.0 req/s: coloc p99 ITL {coloc_p99[-1]:.0f}ms vs disagg {disagg_p99[-1]:.0f}ms; "
      f"p90 TTFT {coloc_ttft[-1]:.0f}ms vs {disagg_ttft[-1]:.0f}ms")
